In [1]:
import sys, os, subprocess, threading
sys.path.append('backend/app/')
import xarray as xr, numpy as np
from backend.app.services import functions
from backend.app.config import PROJECT_DES, DELFT_PATH

In [2]:
# Stream logs
def stream_logs(process, input_dir):
    try:
        for line in process.stdout:
            line = line.strip()
            if not line: continue
            print(line)
    except Exception as e:
        print(f"[INTERNAL ERROR] {e}")
    finally:
        process.wait()
        print("Reorganizing outputs. Please wait...")
        try:
            post_result = functions.postProcess(input_dir)
            print(post_result["message"])
        except Exception as e:
            print(f"Simulation failed: {e}")

In [ ]:
project = '3months'
dir = os.path.join(PROJECT_DES, f"backend/projects/vlnguyen/{project}")
input_dir = os.path.normpath(os.path.join(dir, "input"))
mdu_path = os.path.normpath(os.path.join(input_dir, "FlowFM.mdu"))
bat_path = os.path.normpath(os.path.join(DELFT_PATH, "x64/dflowfm/scripts/run_dflowfm.bat"))
# bat_path = os.path.normpath(os.path.join(DELFT_PATH, "install_fm-suite_2026_02/bin/run_dflowfm.bat"))
# bat_path = os.path.normpath(os.path.join(DELFT_PATH, "install_fm-suite_2025_01/bin/run_dflowfm.bat"))
command = ["cmd.exe", "/c", bat_path, "--autostartstop", mdu_path]

In [ ]:
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding="utf-8", errors="replace", bufsize=1, cwd=input_dir)
threading.Thread(target=stream_logs, args=(process, input_dir), daemon=True).start()

In [2]:
path = r"C:\Users\vanln\Desktop\Grid.nc"
ds = xr.open_dataset(path)
ds

<xarray.Dataset> Size: 212kB
Dimensions:            (mesh2d_nEdges: 3806, Two: 2, mesh2d_nFaces: 2339,
                        mesh2d_nMax_face_nodes: 3, mesh2d_nNodes: 1469)
Coordinates:
    mesh2d_node_x      (mesh2d_nNodes) float32 6kB ...
    mesh2d_node_y      (mesh2d_nNodes) float32 6kB ...
Dimensions without coordinates: mesh2d_nEdges, Two, mesh2d_nFaces,
                                mesh2d_nMax_face_nodes, mesh2d_nNodes
Data variables:
    crs                int32 4B ...
    mesh2d_edge_x      (mesh2d_nEdges) float32 15kB ...
    mesh2d_edge_y      (mesh2d_nEdges) float32 15kB ...
    mesh2d_edge_nodes  (mesh2d_nEdges, Two) int32 30kB ...
    mesh2d_face_nodes  (mesh2d_nFaces, mesh2d_nMax_face_nodes) float32 28kB ...
    mesh2d_edge_faces  (mesh2d_nEdges, Two) int32 30kB ...
    mesh2d_face_x      (mesh2d_nFaces) float32 9kB ...
    mesh2d_face_y      (mesh2d_nFaces) float32 9kB ...
    mesh2d_face_x_bnd  (mesh2d_nFaces, mesh2d_nMax_face_nodes) float32 28kB ...
    mesh2d_face_y_bnd  (mesh2d_nFaces, mesh2d_nMax_face_nodes) float32 28kB ...
    mesh2d             int32 4B ...
    mesh2d_node_z      (mesh2d_nNodes) float32 6kB ...
Attributes:
    institution:  Private
    references:   vanlnNTNU@gmail.com

In [3]:
grid = functions.unstructuredGridCreator(ds)
grid
# grid.to_file('grid.geojson', driver='GeoJSON')

,geometry
0,"POLYGON ((6.48455 62.4831, 6.48595 62.48304, 6..."
1,"POLYGON ((6.48595 62.48304, 6.48585 62.48372, ..."
2,"POLYGON ((6.48455 62.4831, 6.48488 62.48376, 6..."
3,"POLYGON ((6.48595 62.48304, 6.48622 62.48368, ..."
4,"POLYGON ((6.48595 62.48304, 6.48687 62.48302, ..."
...,...
2334,"POLYGON ((6.45798 62.47496, 6.45993 62.47416, ..."
2335,"POLYGON ((6.48248 62.47968, 6.4862 62.47918, 6..."
2336,"POLYGON ((6.46222 62.47459, 6.46403 62.47485, ..."
2337,"POLYGON ((6.45993 62.47416, 6.46222 62.47459, ..."


In [2]:
path = r"backend\projects\vlnguyen\test\output\HYD\FlowFM_his.zarr"
ds = xr.open_zarr(path)
ds

<xarray.Dataset> Size: 361kB
Dimensions:                                  (stations: 5, time: 63,
                                              laydim: 10, cross_section: 1,
                                              cross_section_geom_nNodes: 14,
                                              laydimw: 11,
                                              station_geom_nNodes: 5)
Coordinates:
    station_name                             (stations) |S256 1kB dask.array<chunksize=(5,), meta=np.ndarray>
    station_x_coordinate                     (stations) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    station_y_coordinate                     (stations) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
  * time                                     (time) datetime64[ns] 504B 2024-...
    zcoordinate_c                            (time, stations, laydim) float64 25kB dask.array<chunksize=(63, 5, 10), meta=np.ndarray>
    cross_section_name                       (cross_section) |S256 256B dask.array<chunksize=(1,), meta=np.ndarray>
    zcoordinate_w                            (time, stations, laydimw) float64 28kB dask.array<chunksize=(63, 5, 11), meta=np.ndarray>
    zcoordinate_wu                           (time, stations, laydimw) float64 28kB dask.array<chunksize=(63, 5, 11), meta=np.ndarray>
Dimensions without coordinates: stations, laydim, cross_section,
                                cross_section_geom_nNodes, laydimw,
                                station_geom_nNodes
Data variables: (12/77)
    bedlevel                                 (stations) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    checkerboard_monitor                     (time, laydim) float64 5kB dask.array<chunksize=(63, 10), meta=np.ndarray>
    comp_time                                (time) float64 504B dask.array<chunksize=(63,), meta=np.ndarray>
    Contaminant                              (time, stations, laydim) float64 25kB dask.array<chunksize=(63, 5, 10), meta=np.ndarray>
    cross_section_area                       (time, cross_section) float64 504B dask.array<chunksize=(63, 1), meta=np.ndarray>
    cross_section_Contaminant                (time, cross_section) float64 504B dask.array<chunksize=(63, 1), meta=np.ndarray>
    ...                                       ...
    waterdepth                               (time, stations) float64 3kB dask.array<chunksize=(63, 5), meta=np.ndarray>
    waterlevel                               (time, stations) float64 3kB dask.array<chunksize=(63, 5), meta=np.ndarray>
    wgs84                                    int32 4B ...
    x_velocity                               (time, stations, laydim) float64 25kB dask.array<chunksize=(63, 5, 10), meta=np.ndarray>
    y_velocity                               (time, stations, laydim) float64 25kB dask.array<chunksize=(63, 5, 10), meta=np.ndarray>
    z_velocity                               (time, stations, laydim) float64 25kB dask.array<chunksize=(63, 5, 10), meta=np.ndarray>
Attributes:
    institution:               Deltares
    references:                http://www.deltares.nl
    source:                    Deltares, D-Flow FM Version 1.2.177.142431, Ja...
    history:                   Created on 2026-09-28T15:09:34+0200, D-Flow FM
    date_created:              2026-09-28T15:09:34+0200
    date_modified:             2026-09-28T15:09:34+0200
    Conventions:               CF-1.5 Deltares-0.1 ACDD-1.3
    uuid:                      fb05328e-bd60-af48-bd68-bb22b9fabf30
    time_coverage_start:       2024-05-01T**:00:00+00:00
    time_coverage_end:         2024-06-01T**:00:00+00:00
    time_coverage_duration:    P0000-00-31T00:00:00
    time_coverage_resolution:  P0000-00-00T12:00:00

In [4]:
ds.close()